<a href="https://colab.research.google.com/github/nachopedretti/diploup2026-TP2/blob/main/notebooks/06_prediccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Predicción y archivo de entrega

# 06 – Predicción

En esta última etapa vamos a utilizar el modelo de Regresión Logística
balanceada que seleccionamos durante el entrenamiento y la validación.

El objetivo es generar predicciones para identificar si una persona
es fumadora (1) o no fumadora (0).

Para esto utilizamos el modelo guardado y el umbral de clasificación
de 0.55 que definimos durante la optimización.

In [ ]:
from pathlib import Path
import pandas as pd
import joblib
import urllib.request

base_url = "https://raw.githubusercontent.com/nachopedretti/diploup2026-TP2/main/"

# Datos nuevos, sin etiquetas
X_pred = pd.read_csv(base_url + "data/processed/X_pred.csv")

# Descargar el modelo final
ruta_modelos = Path("/content/models")
ruta_modelos.mkdir(parents=True, exist_ok=True)

ruta_modelo = ruta_modelos / "regresion_logistica_balanceada.pkl"

urllib.request.urlretrieve(
    base_url + "models/regresion_logistica_balanceada.pkl",
    ruta_modelo
)

# Cargar modelo y umbral
modelo_guardado = joblib.load(ruta_modelo)

modelo_final = modelo_guardado["modelo"]
umbral_final = modelo_guardado["umbral"]

# Verificaciones iniciales
print("Dimensiones de X_pred:", X_pred.shape)
print("Modelo:", type(modelo_final).__name__)
print("Umbral:", umbral_final)
print("¿Coinciden las variables?:",
      list(X_pred.columns) == list(modelo_final.feature_names_in_))
print("Valores faltantes:", X_pred.isnull().sum().sum())

Dimensiones de X_pred: (5692, 24)
Modelo: LogisticRegression
Umbral: 0.55
¿Coinciden las variables?: True
Valores faltantes: 0


In [ ]:
import pandas as pd

ruta_original = "https://raw.githubusercontent.com/nachopedretti/diploup2026-TP2/main/data/raw/smoking_prediction_entrega.csv"

df_original = pd.read_csv(ruta_original)

print("Registros originales:", len(df_original))
print("Columnas:", df_original.columns.tolist())
display(df_original.head())


Registros originales: 5692
Columnas: ['ID', 'gender', 'age', 'height(cm)', 'weight(kg)', 'waist(cm)', 'eyesight(left)', 'eyesight(right)', 'hearing(left)', 'hearing(right)', 'systolic', 'relaxation', 'fasting blood sugar', 'Cholesterol', 'triglyceride', 'HDL', 'LDL', 'hemoglobin', 'Urine protein', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'oral', 'dental caries', 'tartar']


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar
0,27358.0,M,25.0,160.0,65.0,3.420139,0.045139,0.002778,0.041667,0.041667,...,3.041667,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0.0,Y
1,27364.0,M,30.0,180.0,80.0,3.458333,0.043056,0.006250,0.041667,0.041667,...,4.208333,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0.0,N
2,27368.0,M,55.0,165.0,60.0,3.416667,0.004861,0.005556,0.041667,0.041667,...,2.041667,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1.0,Y
3,27378.0,M,20.0,175.0,75.0,3.625000,0.045139,0.045139,0.041667,0.041667,...,3.708333,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0.0,N
4,27381.0,M,25.0,165.0,80.0,3.791667,0.043056,0.041667,0.041667,0.041667,...,6.625000,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1.0,Y


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Predicciones del modelo final (Regresión Logística, umbral 0.55)
probabilidades = modelo_final.predict_proba(X_pred)[:, 1]
predicciones_finales = (probabilidades >= umbral_final).astype(int)

# Recuperamos los ID del dataset original
ids = df_original["ID"].astype(int).reset_index(drop=True)

# Verificamos las dimensiones
assert len(ids) == len(predicciones_finales) == 5692

# Generamos el archivo con las columnas solicitadas
df_resultado = pd.DataFrame({
    "ID": ids,
    "smoking_prediction": predicciones_finales
})

# Validaciones
assert df_resultado["ID"].is_unique
assert df_resultado["smoking_prediction"].isin([0, 1]).all()
assert not df_resultado.isna().any().any()

# Carpeta local de resultados
ruta_resultados = Path("/content/results")
ruta_resultados.mkdir(parents=True, exist_ok=True)

# Guardamos el CSV
ruta_salida = ruta_resultados / "smoking_predictions.csv"
df_resultado.to_csv(ruta_salida, index=False)

print("Archivo guardado en:", ruta_salida)
print("Dimensiones:", df_resultado.shape)
print("Columnas:", df_resultado.columns.tolist())
print("Distribución de predicciones:")
print(df_resultado["smoking_prediction"].value_counts())
display(df_resultado.head())


Archivo guardado en: /content/results/smoking_predictions.csv
Dimensiones: (5692, 2)
Columnas: ['ID', 'smoking_prediction']
Distribución de predicciones:
smoking_prediction
1    3231
0    2461
Name: count, dtype: int64


,ID,smoking_prediction
0,27358,1
1,27364,1
2,27368,1
3,27378,1
4,27381,1


In [ ]:

from google.colab import files

files.download("/content/results/smoking_predictions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download(str(ruta_salida))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Conclusión

- Utilizamos la **Regresión Logística balanceada** con umbral **0.55**, seleccionada en la etapa de validación.
- En Test obtuvimos un Accuracy de 72,36%, Recall de 90,23% y F1-score de 70,53%.
- Mantuvimos este modelo por su estabilidad entre Train y Test y menor overfitting que Random Forest.
- Generamos las predicciones de los **5.692 registros**, obteniendo 3.231 fumadores y 2.461 no fumadores.
- Exportamos `smoking_predictions.csv` con las columnas `ID` y `smoking_prediction`.